In [1]:
#Part 1: Paths
# import path and pandas
from pathlib import Path
import pandas as pd

In [9]:
# project root
PROJECT_ROOT = Path.cwd().parent
assert (PROJECT_ROOT / "requirements.txt").exists(), f"Unexpected project root: {PROJECT_ROOT}"

DATA_DIR = PROJECT_ROOT / "data"
PRACTICE_DIR = DATA_DIR / "practice"

PRACTICE_DIR.mkdir(parents=True, exist_ok=True)

print(
    f"data: {DATA_DIR}"
    f"\npractice: {PRACTICE_DIR}"
)

data: c:\Users\Laptop\projects\aldermoor-internal-fraud-analytics\data
practice: c:\Users\Laptop\projects\aldermoor-internal-fraud-analytics\data\practice


In [31]:
p = Path("data/raw/access_logs/access_logs_2025_03.csv")
display(p.name)
display(p.stem)
display(p.suffix)
display(p.parent)

stem_parts = p.stem.split("_")
year = int(stem_parts[-2])
month = int(stem_parts[-1])

print(year, month)

'access_logs_2025_03.csv'

'access_logs_2025_03'

'.csv'

WindowsPath('data/raw/access_logs')

2025 3


In [20]:
# Part 2: Text files
with open(PRACTICE_DIR / "run_notes.txt", "w", encoding="utf-8") as file:
    file.write("run start\n")
    file.write("record received count\n")
    file.write("record rejected count\n")

In [21]:
# 2. appending a new line
with open(PRACTICE_DIR / "run_notes.txt", "a", encoding="utf-8") as file:
    file.write("new record received\n")

In [ ]:
# 3. read the whole file back
with open(PRACTICE_DIR / "run_notes.txt", "r", encoding="utf-8") as file:
    content = file.read()

print(content)

# Nothing happened to the new line appended

run start
record received count
record rejected count
new record received



In [27]:
branches = pd.DataFrame({
    "branch_code": ["00412", "01207", "00098"],
    "branch_name": ["leeds_briggate", "york_coney_street", "hull_whitefriargate"],
    "staff_count": [38, 24, 19],
})

# save to file
branches.to_csv(PRACTICE_DIR / "branches.csv", index=False)

# read it back
branches_reread = pd.read_csv(PRACTICE_DIR / "branches.csv")
display(branches_reread.head())

# the leading zeros of the branch code are gone

# reading it again
branches_read_again = pd.read_csv(PRACTICE_DIR / "branches.csv", dtype={"branch_code": str})
print("_" * 50)
display(branches_read_again.head())

# asserting
assert branches["branch_code"].to_list() == branches_read_again["branch_code"].to_list()


,branch_code,branch_name,staff_count
0,412,leeds_briggate,38
1,1207,york_coney_street,24
2,98,hull_whitefriargate,19


__________________________________________________


,branch_code,branch_name,staff_count
0,00412,leeds_briggate,38
1,01207,york_coney_street,24
2,00098,hull_whitefriargate,19


In [33]:
# Part 4: Partitioned monthly files
monthly_access = {
    1: {"employee_id": ["E00417", "E00588"], "access_events": [880, 610]},
    2: {"employee_id": ["E00417", "E00588", "E00612"], "access_events": [910, 590, 702]},
    3: {"employee_id": ["E00417", "E00612"], "access_events": [1430, 688]},
}

ACCESS_LOGS = PRACTICE_DIR / "access_logs"
ACCESS_LOGS.mkdir(parents=True, exist_ok=True)

# create files
for month, month_data in monthly_access.items():
    month_df = pd.DataFrame(month_data)
    month_path = ACCESS_LOGS / f"access_logs_2025_{month:02d}.csv"
    month_df.to_csv(month_path, index=False)

In [39]:
# Steps 2 to 4: find, read, label, combine and reconcile
log_files = sorted(ACCESS_LOGS.glob("access_logs_*.csv"))

monthly_frames = []
for file in log_files:
    month_df = pd.read_csv(file)
    month_df["month"] = int(file.stem.split("_")[-1])
    print(f"{file.name}: {len(month_df)} rows")
    monthly_frames.append(month_df)

all_access = pd.concat(monthly_frames, ignore_index=True)

expected_rows = sum(len(frame) for frame in monthly_frames)
assert len(all_access) == expected_rows
print(f"Combined: {len(all_access)} rows (expected {expected_rows})")
display(all_access)

access_logs_2025_01.csv: 2 rows
access_logs_2025_02.csv: 3 rows
access_logs_2025_03.csv: 2 rows
Combined: 7 rows (expected 7)


,employee_id,access_events,month
0,E00417,880,1
1,E00588,610,1
2,E00417,910,2
3,E00588,590,2
4,E00612,702,2
5,E00417,1430,3
6,E00612,688,3


In [44]:
# Part 5: Missing files, and Git
def load_csv(path: Path) -> pd.DataFrame:
    """Read a CSV file, raising a clear FileNotFoundErorr
    if it does not exist.
    """
    if not path.exists():
        raise FileNotFoundError(f"Expected data file not found: {path}")
    return pd.read_csv(path)

In [45]:
loaded_branches = load_csv(PRACTICE_DIR / "branches.csv")
assert len(loaded_branches) == 3

missing_path = PRACTICE_DIR /"does_not_exists.csv"
try:
    load_csv(missing_path)
except FileNotFoundError as error:
    print(f"raised as expected: {error}")

else:
    raise AssertionError("Load_csv should have raised FileNotFoundError")

print("load_csv: all test passed")

raised as expected: Expected data file not found: c:\Users\Laptop\projects\aldermoor-internal-fraud-analytics\data\practice\does_not_exists.csv
load_csv: all test passed
